# Oakland corridor activity — second feasibility pilot

Run cells in order in the same Cuebiq Snowflake environment as the raw pilot. This tests the remaining production gates with **provider/country/date partition filters on every source query**. It does not generate full historical outputs. The existing Fruitvale 2024 device-month table is reused. No device IDs are exported.

**What it checks:** provider-specific 2019/2024 coverage, a 2019 corridor match, 2024 HOME block-group assignment and confidence, optional supplied CBG category crosswalk, a bounded block-group sample denominator, all 11 corridors on one day, and a one-month Fruitvale benchmark. Each source query reports wall time. A completed run leaves only small grouped diagnostics in CSV; intermediate device data stays in Snowflake.

**Runtime controls:** Set `RUN_ALL_CORRIDORS_DAY` or `RUN_MONTH_BENCHMARK` to `False` before starting if the one-time cover upload or month scan exceeds your budget. These two gates then remain untested. The source probes use `LIMIT 1` on specific days, not national counts. If a cell takes over 10 minutes, interrupt it and record that gate as unresolved; do not wait 45 minutes. No source query runs over more than one month.

## 1. Connection and controls

In [1]:
import time
from pathlib import Path
import pandas as pd
from IPython.display import display

%load_ext cuebiqmagic.magics
%init_cuebiq_data
snow_engine = get_ipython().user_ns['instance']


schema : args.schema='paas_cda_pe_v3' and database: args.database='cuebiq_data' used
The connection to the Snowflake cluster has been established using database 'cuebiq_data' and schema 'paas_cda_pe_v3'.

You can now retrieve the connection using the following statement:
snow_engine = get_ipython().user_ns['instance']

The connection offers two methods


snow_engine.execute_statement("<SQL Statement>") # execute the SQL statement.


snow_engine.read_sql("<SQL Statement>") # execute the SQL statement and return a Pandas DataFrame.


snow_engine.write_dataframe(dataframe:pd.DataFrame, table_name:str,schema:str, index_in=False, if_exists_in="append",method_in="multi")


In [2]:
SOURCE_SCHEMA = 'CUEBIQ_DATA.PAAS_CDA_PE_V3'
STOP_TABLE = f'{SOURCE_SCHEMA}.STOP_BY_EVENT_DATE_UPLEVELLED'
HOME_TABLE = f'{SOURCE_SCHEMA}.DEVICE_RECURRING_AREA'
TMP_SCHEMA = 'DEDICATED.EDDIT_EAST_BAY_TMP'
PROVIDER_ID, COUNTRY_CODE = 'PAPA', 'US'
PREVIOUS_DEVICE_MONTH = f'{TMP_SCHEMA}.TMP_EB_QUICK_DEVICE_MONTH_PERIOD'
COVER_PATH = Path('./corridors/geohash_level9_oak_bids.csv')
# Optional user-provided CSV: block_group_id, geography, vintage.
# geography: Oakland, San Francisco, Rest East Bay, Rest Bay Area, Rest California, Rest US.
CBG_CROSSWALK_PATH = None
EXPORT_DIR = Path('./activity/pilot2')
RUN_ALL_CORRIDORS_DAY = True
RUN_MONTH_BENCHMARK = True
RUN_SAMPLE_BENCHMARK = True
RUN_EXTRA_YEAR_PROBES = True
DAY_2024 = 20240110
HOME_DATES_2024 = [20240108, 20240114]
HISTORY_DATES_2019 = [20190110, 20190710, 20191210]
MONTH_START, MONTH_END = 20240101, 20240131
MAX_RAW_STOP_MINUTES = 480
P = f'{TMP_SCHEMA}.TMP_EB_P2_'
T = {'COVER':P+'CORRIDOR_GEOHASH9', 'CBG_SEED':P+'CBG_SEED',
     'CBG_SAMPLE':P+'CBG_SAMPLE'}
print('Pilot 2 configured. One-day all-corridor and one-month Fruitvale benchmarks:',
      RUN_ALL_CORRIDORS_DAY, RUN_MONTH_BENCHMARK)

Pilot 2 configured. One-day all-corridor and one-month Fruitvale benchmarks: True True


## 2. Helpers, prior data, and input QA

In [3]:
timings=[]
results={}
def run(label, sql, source=False):
    print(f'Start: {label}', flush=True)
    start=time.monotonic()
    try:
        df=snow_engine.read_sql(sql)
    except Exception:
        timings.append({'step':label,'seconds':round(time.monotonic()-start,1),
                        'source_scan':source,'status':'ERROR'})
        raise
    df.columns=[str(x).lower() for x in df.columns]
    elapsed=round(time.monotonic()-start,1)
    timings.append({'step':label,'seconds':elapsed,'source_scan':source,'status':'OK'})
    print(f'Done: {label} ({elapsed:.1f} s)', flush=True)
    return df

def insert_frame(df, table, columns, chunk_size=4000):
    ddl=', '.join(f'{col} {typ}' for col,typ in columns.items())
    run('Create '+table.rsplit('.',1)[-1],f'CREATE OR REPLACE TABLE {table} ({ddl})')
    start=time.monotonic()
    for offset in range(0,len(df),chunk_size):
        rows=[]
        for row in df.iloc[offset:offset+chunk_size][list(columns)].itertuples(index=False,name=None):
            vals=['NULL' if pd.isna(v) else (str(v) if isinstance(v,(int,float)) and not isinstance(v,bool)
                  else "'"+str(v).replace("'","''")+"'") for v in row]
            rows.append('('+','.join(vals)+')')
        snow_engine.read_sql(f"INSERT INTO {table} ({','.join(columns)}) VALUES "+','.join(rows))
        if (offset//chunk_size+1)%10==0 or offset+chunk_size>=len(df):
            print(f'Uploaded {min(offset+chunk_size,len(df)):,}/{len(df):,} rows',flush=True)
    timings.append({'step':'Upload '+table.rsplit('.',1)[-1],
                    'seconds':round(time.monotonic()-start,1),'source_scan':False,'status':'OK'})
    actual=int(run('Verify '+table.rsplit('.',1)[-1],f'SELECT COUNT(*) AS n FROM {table}').iloc[0]['n'])
    if actual!=len(df): raise AssertionError(f'{table}: {actual} vs {len(df)}')

run('Ensure schema', f'CREATE SCHEMA IF NOT EXISTS {TMP_SCHEMA}')
prior=run('Check 2024 Fruitvale device-months',f"""
SELECT COUNT(*) AS row_count,COUNT(DISTINCT cuebiq_id) AS devices,
       COUNT(DISTINCT corridor_id) AS corridors
FROM {PREVIOUS_DEVICE_MONTH}
WHERE year_month=202401 AND time_period='all'
""")
display(prior)
if int(prior.iloc[0]['devices'])==0:
    raise ValueError('Previous pilot device-month table has no January 2024 devices. Re-run pilot (6) first.')

if not COVER_PATH.exists():
    alternate=Path('./geo/geohash_level9_oak_bids.csv')
    if alternate.exists(): COVER_PATH=alternate
if not COVER_PATH.exists():
    raise FileNotFoundError(f'Set COVER_PATH to the complete 11-corridor geohash9 CSV: {COVER_PATH}')
cover=pd.read_csv(COVER_PATH,dtype={'corridor_id':str,'corridor_name':str,'geohash9':str})
required={'corridor_id','corridor_name','geohash9','coverage_fraction'}
if required-set(cover): raise ValueError(f'Cover missing {required-set(cover)}')
cover['corridor_id']='Oakland BID:'+cover['corridor_id'].str.strip()
cover['geohash9']=cover['geohash9'].str.lower().str.strip()
cover['coverage_fraction']=pd.to_numeric(cover['coverage_fraction'])
if cover.duplicated(['corridor_id','geohash9']).any(): raise ValueError('Duplicate corridor-cell pairs')
if not cover.geohash9.str.len().eq(9).all() or not cover.coverage_fraction.between(.8,1).all():
    raise ValueError('Expected 9-character cells at coverage >= 0.8')
if cover.corridor_id.nunique()!=11: raise ValueError(f'Expected 11 corridors, found {cover.corridor_id.nunique()}')
fruit=cover.loc[cover.corridor_name.eq('Fruitvale'),'corridor_id'].drop_duplicates()
if len(fruit)!=1: raise ValueError('Fruitvale missing or ambiguous')
FRUITVALE_ID=fruit.iloc[0].replace("'","''")
print(f'Cover: {len(cover):,} corridor-cell pairs, {cover.geohash9.nunique():,} distinct cells, '
      f'{int((cover.groupby("geohash9").corridor_id.nunique()>1).sum()):,} shared cells')

Start: Ensure schema
Done: Ensure schema (1.4 s)
Start: Check 2024 Fruitvale device-months
Done: Check 2024 Fruitvale device-months (0.9 s)


,row_count,devices,corridors
0,3537,3537,1


Cover: 218,259 corridor-cell pairs, 218,021 distinct cells, 238 shared cells


## 3. Exact-date source availability

Each query touches one partition and exits on its first matching provider/country row. A `found=0` on three dates does not prove the whole year is absent; it identifies dates requiring further investigation. The 2024 control helps distinguish an empty partition from a query issue.

In [4]:
probe_dates=HISTORY_DATES_2019+[DAY_2024]
if RUN_EXTRA_YEAR_PROBES: probe_dates += [20200110,20250110]
availability=[]
for asset,table,date_col in [('stops',STOP_TABLE,'event_date'),
                              ('home',HOME_TABLE,'snapshot_event_date')]:
    for day in probe_dates:
        tag_filter="AND tag_type_code='HOME'" if asset=='home' else ''
        q=f"""SELECT {date_col} AS sample_date FROM {table}
        WHERE provider_id='{PROVIDER_ID}' AND country_code='{COUNTRY_CODE}'
          AND {date_col}={day} {tag_filter}
        LIMIT 1"""
        hit=run(f'{asset} availability {day}',q,source=True)
        availability.append({'asset':asset,'date':day,'found':int(len(hit)>0)})
results['availability']=pd.DataFrame(availability)
display(results['availability'])

Start: stops availability 20190110
Done: stops availability 20190110 (3.5 s)
Start: stops availability 20190710
Done: stops availability 20190710 (5.8 s)
Start: stops availability 20191210
Done: stops availability 20191210 (3.7 s)
Start: stops availability 20240110
Done: stops availability 20240110 (4.6 s)
Start: stops availability 20200110
Done: stops availability 20200110 (3.0 s)
Start: stops availability 20250110
Done: stops availability 20250110 (4.4 s)
Start: home availability 20190110
Done: home availability 20190110 (3.4 s)
Start: home availability 20190710
Done: home availability 20190710 (4.2 s)
Start: home availability 20191210
Done: home availability 20191210 (3.2 s)
Start: home availability 20240110
Done: home availability 20240110 (4.0 s)
Start: home availability 20200110
Done: home availability 20200110 (4.3 s)
Start: home availability 20250110
Done: home availability 20250110 (6.6 s)


,asset,date,found
0,stops,20190110,0
1,stops,20190710,0
2,stops,20191210,0
3,stops,20240110,1
4,stops,20200110,0
5,stops,20250110,1
6,home,20190110,0
7,home,20190710,0
8,home,20191210,0
9,home,20240110,1


## 4. US HOME match and block-group quality

Uses only the January 2024 observed Fruitvale devices from pilot 1. Both HOME source dates are explicit snapshot partitions. The output separates any HOME, populated US block group, and ≥0.60 confidence. It also compares the two snapshot dates; production must choose a deterministic snapshot rule for each month.

In [5]:
home_day_pred=' OR '.join(f'h.snapshot_event_date={d}' for d in HOME_DATES_2024)
q=f"""
WITH devices AS (
 SELECT DISTINCT cuebiq_id FROM {PREVIOUS_DEVICE_MONTH}
 WHERE year_month=202401 AND time_period='all' AND corridor_id='{FRUITVALE_ID}'
), h AS (
 SELECT d.cuebiq_id,h.snapshot_event_date,h.block_group_id,h.geohash_id,
        h.confidence_level,
        ROW_NUMBER() OVER (PARTITION BY d.cuebiq_id,h.snapshot_event_date
             ORDER BY h.confidence_level DESC NULLS LAST,h.block_group_id) AS rn
 FROM devices d JOIN {HOME_TABLE} h ON h.cuebiq_id=d.cuebiq_id
 WHERE h.provider_id='{PROVIDER_ID}' AND h.country_code='{COUNTRY_CODE}'
   AND h.tag_type_code='HOME' AND ({home_day_pred})
), chosen AS (SELECT * FROM h WHERE rn=1)
SELECT d.cuebiq_id,c.snapshot_event_date,c.block_group_id,c.geohash_id,c.confidence_level
FROM devices d LEFT JOIN chosen c ON d.cuebiq_id=c.cuebiq_id
"""
# Do not display or export row-level device data.
assigned=run('HOME assignment for observed devices',q,source=True)
all_ids=int(prior.iloc[0]['devices'])
summary=[]
for day in HOME_DATES_2024:
    # The left join has one NULL-date row only when a device has neither snapshot.
    x=assigned.loc[assigned.snapshot_event_date.eq(day)]
    valid=x.block_group_id.astype('string').str.fullmatch(r'\d{12}').fillna(False)
    summary.append({'snapshot_date':day,'observed_devices':all_ids,
        'devices_with_home':x.cuebiq_id.nunique(),
        'devices_with_12_digit_cbg':int(valid.sum()),
        'devices_confident_cbg':int((valid & x.confidence_level.ge(.60).fillna(False)).sum()),
        'non_null_geohash':int(x.geohash_id.notna().sum()),
        'distinct_cbgs':x.loc[valid,'block_group_id'].nunique()})
results['home_match']=pd.DataFrame(summary)
display(results['home_match'])
matched=assigned.dropna(subset=['snapshot_event_date']).copy()
pair=matched.pivot(index='cuebiq_id',columns='snapshot_event_date',values='block_group_id')
if set(HOME_DATES_2024).issubset(pair.columns):
    both=pair[HOME_DATES_2024].dropna()
    results['home_stability']=pd.DataFrame([{'matched_both':len(both),
        'same_cbg':int(both.iloc[:,0].eq(both.iloc[:,1]).sum())}])
    display(results['home_stability'])
# Hold CBG values in memory for optional crosswalk and a bounded denominator benchmark.
cbg_series=matched.loc[matched.block_group_id.astype('string').str.fullmatch(r'\d{12}').fillna(False),
                       'block_group_id'].astype(str)
print('Distinct observed CBGs:',cbg_series.nunique())

Start: HOME assignment for observed devices
Done: HOME assignment for observed devices (216.1 s)


,snapshot_date,observed_devices,devices_with_home,devices_with_12_digit_cbg,devices_confident_cbg,non_null_geohash,distinct_cbgs
0,20240108,3537,2370,0,0,0,0
1,20240114,3537,2910,0,0,0,0


,matched_both,same_cbg
0,2360,1773


Distinct observed CBGs: 0


## 5. Geography crosswalk gate

US HOME has a Census block group, not a home geohash. A block-group-to-category lookup is required for the six requested origin categories. Supply `CBG_CROSSWALK_PATH` (with `block_group_id`, `geography`, and `vintage`) to measure match and validate labels. Without it, the notebook reports this as an open production input. A CBG centroid/geometry is separately needed for home-to-corridor distance.

In [6]:
categories={'Oakland','San Francisco','Rest East Bay','Rest Bay Area',
            'Rest California','Rest US'}
if CBG_CROSSWALK_PATH is None:
    results['geography_gate']=pd.DataFrame([{'status':'MISSING_CROSSWALK',
        'observed_cbgs':cbg_series.nunique(),'matched_cbgs':0,
        'vintage':'UNKNOWN'}])
else:
    xwalk=pd.read_csv(CBG_CROSSWALK_PATH,dtype={'block_group_id':str,'geography':str,'vintage':str})
    need={'block_group_id','geography','vintage'}
    if need-set(xwalk): raise ValueError(f'Crosswalk missing {need-set(xwalk)}')
    if xwalk.block_group_id.duplicated().any() or not xwalk.block_group_id.str.fullmatch(r'\d{12}').all():
        raise ValueError('Crosswalk must have one valid 12-digit ID per block group')
    if not set(xwalk.geography).issubset(categories):
        raise ValueError(f'Unexpected categories: {set(xwalk.geography)-categories}')
    observed=pd.DataFrame({'block_group_id':cbg_series.drop_duplicates()})
    joined=observed.merge(xwalk,on='block_group_id',how='left')
    results['geography_gate']=pd.DataFrame([{'status':'CHECKED',
        'observed_cbgs':len(joined),'matched_cbgs':joined.geography.notna().sum(),
        'vintage':','.join(sorted(xwalk.vintage.dropna().unique()))}])
display(results['geography_gate'])

,status,observed_cbgs,matched_cbgs,vintage
0,MISSING_CROSSWALK,0,0,UNKNOWN


## 6. Sample denominator method and cost

This is a **bounded scan of one HOME snapshot partition**, using up to 25 CBGs observed among visitors. The denominator is all distinct eligible PAPA devices with a confident HOME in those CBGs on that date. It checks feasibility and cost only: these visitor-derived CBGs cannot normalize all Oakland/Bay Area travel, and 2019 reference values require 2019 coverage plus a complete region crosswalk. No device IDs leave Snowflake.

In [7]:
if RUN_SAMPLE_BENCHMARK and cbg_series.nunique():
    top=cbg_series.value_counts().head(25).index.tolist()
    seed=pd.DataFrame({'block_group_id':top})
    insert_frame(seed,T['CBG_SEED'],{'block_group_id':'VARCHAR'})
    results['sample_denominator']=run('One-day 25-CBG sample denominator',f"""
    SELECT h.block_group_id,COUNT(DISTINCT h.cuebiq_id) AS sample_devices,
           COUNT(*) AS home_rows
    FROM {HOME_TABLE} h JOIN {T['CBG_SEED']} c
      ON h.block_group_id=c.block_group_id
    WHERE h.provider_id='{PROVIDER_ID}' AND h.country_code='{COUNTRY_CODE}'
      AND h.snapshot_event_date={HOME_DATES_2024[0]}
      AND h.tag_type_code='HOME' AND h.confidence_level>=0.60
    GROUP BY h.block_group_id
    """,source=True)
    display(results['sample_denominator'].describe(include='all'))
    hist=results['availability'].query("asset == 'home' and date < 20200101 and found == 1")['date'].tolist()
    if hist:
        historical_day=int(hist[0])
        results['sample_denominator_2019']=run(f'2019 same-CBG sample denominator {historical_day}',f"""
        SELECT h.block_group_id,COUNT(DISTINCT h.cuebiq_id) AS sample_devices
        FROM {HOME_TABLE} h JOIN {T['CBG_SEED']} c ON h.block_group_id=c.block_group_id
        WHERE h.provider_id='{PROVIDER_ID}' AND h.country_code='{COUNTRY_CODE}'
          AND h.snapshot_event_date={historical_day}
          AND h.tag_type_code='HOME' AND h.confidence_level>=0.60
        GROUP BY h.block_group_id
        """,source=True)
        display(results['sample_denominator_2019'].describe(include='all'))
    else: print('2019 denominator comparison unavailable on probed dates.')
else:
    print('Sample benchmark skipped; sample denominator remains untested.')

Sample benchmark skipped; sample denominator remains untested.


## 7. One-time complete cover upload, then all 11 corridors on one day

All 218k or so corridor-cell pairs are uploaded once to a separate pilot table. If this table already has the expected count and per-corridor counts, it is reused. The join query reads **one 2024 event-date partition**; geohash9 membership and row counts validate geography and reveal overlap. Distinct devices per corridor/day are a preliminary visit measure. Times for upload and source scan are recorded separately.

In [8]:
if RUN_ALL_CORRIDORS_DAY:
    try:
        existing=run('Check complete cover',f"""SELECT corridor_id,COUNT(*) AS n
          FROM {T['COVER']} GROUP BY corridor_id""")
        expected=cover.groupby('corridor_id').size().to_dict()
        current=dict(zip(existing.corridor_id.astype(str),existing.n.astype(int)))
        reuse=(current==expected)
    except Exception:
        reuse=False
    if not reuse:
        insert_frame(cover[['corridor_id','corridor_name','geohash9','coverage_fraction']],T['COVER'],
          {'corridor_id':'VARCHAR','corridor_name':'VARCHAR',
           'geohash9':'VARCHAR','coverage_fraction':'FLOAT'})
    else:
        print('Reusing complete 11-corridor cover.')
    results['corridor_day']=run('All-corridor one-day scan',f"""
    SELECT c.corridor_id,c.corridor_name,COUNT(*) AS matched_stop_slices,
           COUNT(DISTINCT s.cuebiq_id) AS distinct_devices,
           COUNT(DISTINCT s.geohash_id) AS active_cells
    FROM {STOP_TABLE} s JOIN {T['COVER']} c ON s.geohash_id=c.geohash9
    WHERE s.provider_id='{PROVIDER_ID}' AND s.country_code='{COUNTRY_CODE}'
      AND s.event_date={DAY_2024} AND s.stop_zoned_datetime IS NOT NULL
      AND s.dwell_time_minutes>0 AND s.dwell_time_minutes<={MAX_RAW_STOP_MINUTES}
    GROUP BY c.corridor_id,c.corridor_name ORDER BY c.corridor_name
    """,source=True)
    display(results['corridor_day'])
else:
    print('All-corridor benchmark skipped.')

Start: Check complete cover
Start: Create TMP_EB_P2_CORRIDOR_GEOHASH9
Done: Create TMP_EB_P2_CORRIDOR_GEOHASH9 (0.6 s)
Uploaded 40,000/218,259 rows
Uploaded 80,000/218,259 rows
Uploaded 120,000/218,259 rows
Uploaded 160,000/218,259 rows
Uploaded 200,000/218,259 rows
Uploaded 218,259/218,259 rows
Start: Verify TMP_EB_P2_CORRIDOR_GEOHASH9
Done: Verify TMP_EB_P2_CORRIDOR_GEOHASH9 (0.4 s)
Start: All-corridor one-day scan
Done: All-corridor one-day scan (34.0 s)


,corridor_id,corridor_name,matched_stop_slices,distinct_devices,active_cells
0,Oakland BID:2,Chinatown,653,513,590
1,Oakland BID:1,Downtown,1181,884,881
2,Oakland BID:8,Fruitvale,684,566,459
3,Oakland BID:10,Jack London,797,546,722
4,Oakland BID:4,Koreatown/Northgate,476,359,356
5,Oakland BID:3,Lake Merritt,1136,764,933
6,Oakland BID:9,Lakeshore,134,109,97
7,Oakland BID:11,Laurel,145,135,124
8,Oakland BID:5,Montclair,70,62,62
9,Oakland BID:6,Rockridge,190,156,170


## 8. Conditional 2019 corridor match and one-month benchmark

Run a one-day 2019 corridor match only if a 2019 PAPA/US stop partition was found. The month query uses the existing Fruitvale cover and a direct January 2024 event-date predicate. Its device-day aggregation approximates production work; it is an elapsed-time benchmark, not a publishable monthly statistic. Event-date slices around midnight and repeat visitor logic still need the production dedup and local-day rules from pilot 1.

In [9]:
if RUN_ALL_CORRIDORS_DAY:
    candidates=results['availability'].query("asset == 'stops' and date < 20200101 and found == 1")['date'].tolist()
    if candidates:
        day=int(candidates[0])
        results['historical_corridor']=run(f'2019 corridor match {day}',f"""
        SELECT c.corridor_id,c.corridor_name,COUNT(*) AS matched_stop_slices
        FROM {STOP_TABLE} s JOIN {T['COVER']} c ON s.geohash_id=c.geohash9
        WHERE s.provider_id='{PROVIDER_ID}' AND s.country_code='{COUNTRY_CODE}'
          AND s.event_date={day}
        GROUP BY 1,2 ORDER BY 2
        """,source=True)
        display(results['historical_corridor'])
    else: print('No 2019 PAPA/US stop row found on probed dates; 2019 corridor match unresolved.')

if RUN_MONTH_BENCHMARK:
    # Existing single-corridor cover from pilot 1, avoiding the complete-cover upload dependency.
    old_cover=f'{TMP_SCHEMA}.TMP_EB_PILOT_CORRIDOR_GEOHASH9'
    results['month_benchmark']=run('Fruitvale one-month device-day scan',f"""
    WITH dedup AS (
      SELECT s.cuebiq_id,s.geohash_id,s.stop_zoned_datetime,s.dwell_time_minutes,
        ROW_NUMBER() OVER (PARTITION BY s.cuebiq_id,s.geohash_id,
          s.stop_zoned_datetime,s.dwell_time_minutes
          ORDER BY s.processing_date DESC,s.event_date ASC) AS rn
      FROM {STOP_TABLE} s JOIN {old_cover} c ON s.geohash_id=c.geohash9
      WHERE s.provider_id='{PROVIDER_ID}' AND s.country_code='{COUNTRY_CODE}'
        AND s.event_date BETWEEN {MONTH_START} AND {MONTH_END}
        AND c.corridor_id='{FRUITVALE_ID}'
        AND s.stop_zoned_datetime IS NOT NULL
        AND s.dwell_time_minutes>0 AND s.dwell_time_minutes<={MAX_RAW_STOP_MINUTES}
    ), days AS (
      SELECT cuebiq_id,TO_CHAR(TO_TIMESTAMP_TZ(stop_zoned_datetime),'YYYYMMDD') AS local_day,
             COUNT(*) AS raw_stops
      FROM dedup WHERE rn=1 GROUP BY 1,2
    )
    SELECT SUM(raw_stops) AS raw_stops,COUNT(*) AS device_days,
           COUNT(DISTINCT cuebiq_id) AS distinct_devices FROM days
    """,source=True)
    display(results['month_benchmark'])
else: print('Month benchmark skipped.')

No 2019 PAPA/US stop row found on probed dates; 2019 corridor match unresolved.
Start: Fruitvale one-month device-day scan
Done: Fruitvale one-month device-day scan (930.1 s)


,raw_stops,device_days,distinct_devices
0,29699,23796,17954


## 9. Save grouped evidence and decision checklist

The 2019 baseline is feasible only if provider-specific 2019 stops **and** usable HOME block groups exist. To make production geography and normalization, supply a validated CBG crosswalk with boundary vintage, test its visitor match rate, and benchmark the region-wide sample denominator. A 25-CBG benchmark alone does not establish nationwide or Bay Area scale. Use Snowflake query history for bytes, partitions, and warehouse credits for the three timed source benchmarks.

In [10]:
EXPORT_DIR.mkdir(parents=True,exist_ok=True)
results['timings']=pd.DataFrame(timings)
for name,df in results.items():
    df.to_csv(EXPORT_DIR/f'{name}.csv',index=False)
display(results['timings'])
avail=results['availability']
status=pd.DataFrame([
 {'gate':'2019 stop provider dates','result':'FOUND' if avail.query("asset=='stops' and date<20200101 and found==1").shape[0] else 'UNRESOLVED'},
 {'gate':'2019 HOME provider dates','result':'FOUND' if avail.query("asset=='home' and date<20200101 and found==1").shape[0] else 'UNRESOLVED'},
 {'gate':'2024 HOME block group match','result':'MEASURED'},
 {'gate':'CBG category crosswalk','result':results['geography_gate'].iloc[0]['status']},
 {'gate':'CBG sample denominator cost','result':'MEASURED' if 'sample_denominator' in results else 'UNTESTED'},
 {'gate':'11-corridor one-day cost','result':'MEASURED' if 'corridor_day' in results else 'UNTESTED'},
 {'gate':'one-month Fruitvale cost','result':'MEASURED' if 'month_benchmark' in results else 'UNTESTED'},
])
status.to_csv(EXPORT_DIR/'decision_checklist.csv',index=False)
display(status)
print('Grouped diagnostic files:',EXPORT_DIR.resolve())

,step,seconds,source_scan,status
0,Ensure schema,1.4,False,OK
1,Check 2024 Fruitvale device-months,0.9,False,OK
2,stops availability 20190110,3.5,True,OK
3,stops availability 20190710,5.8,True,OK
4,stops availability 20191210,3.7,True,OK
5,stops availability 20240110,4.6,True,OK
6,stops availability 20200110,3.0,True,OK
7,stops availability 20250110,4.4,True,OK
8,home availability 20190110,3.4,True,OK
9,home availability 20190710,4.2,True,OK


,gate,result
0,2019 stop provider dates,UNRESOLVED
1,2019 HOME provider dates,UNRESOLVED
2,2024 HOME block group match,MEASURED
3,CBG category crosswalk,MISSING_CROSSWALK
4,CBG sample denominator cost,UNTESTED
5,11-corridor one-day cost,MEASURED
6,one-month Fruitvale cost,MEASURED


Grouped diagnostic files: /home/jovyan/eddit-east-bay/activity/pilot2
